## Required Imports

In [ ]:
# notebooks/07_batter_data_collection.ipynb

import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from pybaseball import statcast_batter, playerid_lookup
from pybaseball import cache
cache.enable()

print("Batter data collection initialized")

## Define Batter Pool

In [ ]:
# Load your existing pitcher data to find frequent opponents
from pitch_analysis import load_clean_data
pitcher_data = load_clean_data()

# Identify batters who appeared most frequently against your pitchers
# The 'batter' column contains the batter MLBAM ID
batter_counts = pitcher_data['batter'].value_counts()

print("=== Most Frequent Batters in Dataset ===")
print(f"Unique batters faced: {len(batter_counts)}")
print(f"\nTop 30 most frequently faced:")
print(batter_counts.head(30))

# Select top 50 batters by frequency for analysis
# These are the batters with most exposure to your pitcher pool
top_batter_ids = batter_counts.head(50).index.tolist()
print(f"\nSelected {len(top_batter_ids)} batters for analysis")

## Verify Batter Columns

In [ ]:
# Confirm batter column and check what other batter info is available
print("=== Batter-Related Columns in Pitcher Data ===")
batter_cols = [col for col in pitcher_data.columns 
               if 'batter' in col.lower() or 
                  'bat' in col.lower() or
                  'stand' in col.lower()]
print(batter_cols)

# Preview batter data already in pitcher dataset
print("\n=== Sample Batter Records ===")
print(pitcher_data[['batter', 'stand', 'pitch_type', 
                     'events', 'description']].head(10))

## Extract Batter Performance from Existing Data

In [ ]:
# Extract batter performance against each pitch type from existing data
# This avoids a large additional data pull and uses what you already have

batter_vs_pitch = pitcher_data.groupby(
    ['batter', 'stand', 'pitch_type']
).agg(
    pitches_seen=('pitch_type', 'count'),
    swinging_strikes=('description', lambda x: 
                      (x == 'swinging_strike').sum()),
    called_strikes=('description', lambda x: 
                    (x == 'called_strike').sum()),
    balls_in_play=('events', lambda x: 
                   x.isin(['single', 'double', 'triple', 
                           'home_run', 'field_out', 'grounded_into_double_play',
                           'force_out', 'sac_fly']).sum()),
    hits=('events', lambda x: 
          x.isin(['single', 'double', 'triple', 'home_run']).sum()),
    strikeouts=('events', lambda x: 
                (x == 'strikeout').sum()),
    walks=('events', lambda x: (x == 'walk').sum()),
    home_runs=('events', lambda x: (x == 'home_run').sum())
).reset_index()

# Calculate rates
batter_vs_pitch['whiff_rate'] = (
    batter_vs_pitch['swinging_strikes'] / 
    batter_vs_pitch['pitches_seen']
).round(3)

batter_vs_pitch['contact_rate'] = (
    batter_vs_pitch['balls_in_play'] / 
    batter_vs_pitch['pitches_seen']
).round(3)

batter_vs_pitch['hit_rate'] = (
    batter_vs_pitch['hits'] / 
    batter_vs_pitch['pitches_seen'].clip(lower=1)
).round(3)

# Filter to batters with meaningful sample size
batter_vs_pitch = batter_vs_pitch[
    batter_vs_pitch['pitches_seen'] >= 20
]

print(f"=== Batter vs Pitch Type Performance ===")
print(f"Batter-pitch combinations with 20+ pitches: {len(batter_vs_pitch)}")
print(f"Unique batters: {batter_vs_pitch['batter'].nunique()}")
print(f"\nSample records:")
print(batter_vs_pitch.head(10).to_string(index=False))

## Pull Batter Names

In [ ]:
from pybaseball import playerid_reverse_lookup

# Get names for all batters in your dataset
unique_batter_ids = batter_vs_pitch['batter'].unique().tolist()

print(f"Looking up names for {len(unique_batter_ids)} batters...")

try:
    batter_names = playerid_reverse_lookup(
        unique_batter_ids, 
        key_type='mlbam'
    )
    batter_names = batter_names[['key_mlbam', 'name_first', 'name_last']].copy()
    batter_names['batter_name'] = (batter_names['name_first'] + 
                                    ' ' + batter_names['name_last'])
    batter_names = batter_names.rename(columns={'key_mlbam': 'batter'})
    
    # Merge names into batter performance data
    batter_vs_pitch = batter_vs_pitch.merge(
        batter_names[['batter', 'batter_name']], 
        on='batter', 
        how='left'
    )
    
    print(f"Names resolved for {batter_names.shape[0]} batters")
    print("\nSample with names:")
    print(batter_vs_pitch[['batter_name', 'stand', 'pitch_type', 
                            'pitches_seen', 'whiff_rate', 
                            'hit_rate']].head(10).to_string(index=False))

except Exception as e:
    print(f"Name lookup error: {e}")
    print("Proceeding with batter IDs only")
    batter_vs_pitch['batter_name'] = batter_vs_pitch['batter'].astype(str)

## Build Batter Vulnerability Profiles

In [ ]:
# For each batter calculate their vulnerability scores by pitch type
# Vulnerability = high whiff rate + low hit rate against a pitch type

# Normalize metrics to 0-1 scale for scoring
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Pivot to get per-batter metrics across pitch types
batter_pivot = batter_vs_pitch.pivot_table(
    index=['batter', 'batter_name', 'stand'],
    columns='pitch_type',
    values=['whiff_rate', 'hit_rate', 'pitches_seen']
).reset_index()

# Flatten column names
batter_pivot.columns = [
    '_'.join(col).strip('_') if col[1] else col[0] 
    for col in batter_pivot.columns
]

print(f"=== Batter Vulnerability Profiles ===")
print(f"Batters profiled: {len(batter_pivot)}")
print(f"Columns available: {list(batter_pivot.columns)}")

## Calculate Pitch Type Vulnerability Scores

In [ ]:
# For each pitch type calculate a vulnerability score per batter
# Higher score = batter is more vulnerable to that pitch type

pitch_types = batter_vs_pitch['pitch_type'].unique()
vulnerability_records = []

for _, batter_row in batter_vs_pitch.iterrows():
    # Vulnerability combines whiff rate and inverse hit rate
    # Normalized so both contribute equally
    vulnerability_score = (
        batter_row['whiff_rate'] * 0.6 +      # whiff rate weighted higher
        (1 - batter_row['hit_rate']) * 0.4     # inverse hit rate
    ) * 100

    vulnerability_records.append({
        'batter': batter_row['batter'],
        'batter_name': batter_row['batter_name'],
        'stand': batter_row['stand'],
        'pitch_type': batter_row['pitch_type'],
        'pitches_seen': batter_row['pitches_seen'],
        'whiff_rate': batter_row['whiff_rate'],
        'hit_rate': batter_row['hit_rate'],
        'vulnerability_score': round(vulnerability_score, 2)
    })

vulnerability_df = pd.DataFrame(vulnerability_records)

print("=== Vulnerability Score Distribution ===")
print(vulnerability_df.groupby('pitch_type')['vulnerability_score']
      .describe().round(2))

## Save Batter Data

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

batter_vs_pitch.to_parquet(
    '../data/processed/batter_vs_pitch.parquet',
    index=False
)

vulnerability_df.to_parquet(
    '../data/processed/batter_vulnerability.parquet',
    index=False
)

print("=== Batter Data Saved ===")
print(f"batter_vs_pitch.parquet: {len(batter_vs_pitch)} records")
print(f"batter_vulnerability.parquet: {len(vulnerability_df)} records")